<a href="https://colab.research.google.com/github/ijazkhan0351-bot/Ijazweek1-ml-assignment/blob/main/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ijazkhan0351-bot/Ijazweek1-ml-assignment/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
%pip install -q duckdb huggingface_hub

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = "hf://datasets/FlyRank/internship-warehouse"
print("Connected.")

Connected.


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Signal 1 (flag-linked): CTR vs. position** — this is the signal behind
FlyRank's CTR-fix logic. Hypothesis: better (lower-numbered) average
position should correlate with higher CTR.

**Signal 2: Impression volume vs. CTR** — the signal behind quick-win
logic. Hypothesis: pages with high impression volume but low CTR are
strong quick-win candidates (fixing them yields the biggest absolute
click gain).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df = con.sql(f"""
    SELECT
        content_hash_id,
        AVG(gsc_impressions) as avg_impressions,
        AVG(gsc_clicks) as avg_clicks,
        AVG(gsc_avg_position) as avg_position,
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) as ctr
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()

print(f"Total pages in this slice: {len(df)}")

# --- Signal 1: CTR vs position bucket table ---
import pandas as pd
df["position_bucket"] = pd.cut(df["avg_position"], bins=[0, 3, 10, 20, 1000],
                                 labels=["1-3", "4-10", "11-20", "21+"])
signal1_table = df.groupby("position_bucket", observed=True).agg(
    n=("content_hash_id", "count"),
    mean_ctr=("ctr", "mean")
).reset_index()
print("\n--- Signal 1: CTR by position bucket ---")
print(signal1_table)

# --- Signal 2: CTR by impression-volume bucket ---
df["volume_bucket"] = pd.qcut(df["avg_impressions"], q=4, duplicates="drop",
                                labels=["low", "medium", "high", "very high"])
signal2_table = df.groupby("volume_bucket", observed=True).agg(
    n=("content_hash_id", "count"),
    mean_ctr=("ctr", "mean")
).reset_index()
print("\n--- Signal 2: CTR by impression-volume bucket ---")
print(signal2_table)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total pages in this slice: 176738

--- Signal 1: CTR by position bucket ---
  position_bucket      n  mean_ctr
0             1-3  16144  0.010589
1            4-10  81988  0.004926
2           11-20  32203  0.003211
3             21+  44969  0.001928

--- Signal 2: CTR by impression-volume bucket ---
  volume_bucket      n  mean_ctr
0           low  46070  0.009929
1        medium  42303  0.002675
2          high  44189  0.002408
3     very high  44176  0.003055


**Signal 1 verdict: [CONFIRMED / OPPOSITE / MIXED / FALSE]**
[One line: does mean_ctr clearly drop as position_bucket gets worse? If yes
→ CONFIRMED. If CTR is flat or reversed → MIXED/OPPOSITE.]

**Signal 2 verdict: [CONFIRMED / OPPOSITE / MIXED / FALSE]**
[One line: do high-volume pages actually show lower CTR (opportunity), or
is it flat/reversed? Write exactly what the table shows — a clean negative
here is a valid, useful finding.]

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

**The rule:** Score = normalized impression volume × (1 - normalized CTR),
weighted toward pages with real traffic potential but underperforming CTR.
One reason code and one action label are assigned per page based on which
threshold it crosses.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import numpy as np

# Normalize signals 0-1
df["vol_norm"] = (df["avg_impressions"] - df["avg_impressions"].min()) / \
                  (df["avg_impressions"].max() - df["avg_impressions"].min())
df["ctr_gap"] = 1 - (df["ctr"] / df["ctr"].max())  # bigger gap = more room to improve

df["score"] = df["vol_norm"] * df["ctr_gap"]

def assign_reason_and_action(row):
    if row["avg_impressions"] > df["avg_impressions"].quantile(0.75) and row["ctr"] < df["ctr"].median():
        return "high_volume_low_ctr", "quick_win_ctr_fix"
    elif row["avg_position"] > 20:
        return "poor_position", "review_ranking"
    else:
        return "low_priority", "monitor"

df[["reason_code", "action_label"]] = df.apply(
    lambda r: pd.Series(assign_reason_and_action(r)), axis=1
)

queue = df[["content_hash_id", "score", "reason_code", "action_label",
            "avg_impressions", "avg_position", "ctr"]].sort_values("score", ascending=False)

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Queue written: {len(queue)} rows")
queue.head(10)

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top10 = queue.head(10)
print(top10.to_string(index=False))

For each of the top 10 (page 1 through 10 by rank):

1. **[content_hash_id]** — action: [action_label]. Why: [one line, e.g.
   "high impressions, CTR well below median for its position"]. What would
   make this wrong: [e.g. "if this page's low CTR is actually correct
   because it targets a low-intent query"].
2. [repeat same format for rows 2-10]

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
The weakest pick in my top 10 is [row #] — its score is high mainly because
of volume, but its CTR gap is small, meaning there may not be much real
room for improvement. This is a case where the rule's simplicity (multiplying
two signals) could over-rank a page that doesn't actually need action.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.